# Fase 3 — Modelado | Actividad 02: GC1 — SARIMA y Prophet (Limón Sutil y Dulce)

**Fase:** 3 — Modelado (v2 Reentrenamiento)
**Grupo de control 1 (GC1):** modelos **univariados** (solo la serie de producción propia),
sin regresores externos. Referencia clave: comparación contra el baseline **Naive** (Act. 01).

---

## Plan aprobado (6 verificaciones)

1. **Split** — estricto cronológico: train `2016-07..2023-12` (90m), val `2024` (12m),
   test `2025` (12m). Sin splitteo aleatorio ni k-fold. Buffer `2016-01..2016-06` (historia,
   no evaluado).
2. **Semilla** — `seed=42` global antes de cada `fit`. SARIMAX (MLE statsmodels) es
   determinista; Prophet con `mcmc_samples=0` (puntos MAP, sin MCMC) para eliminar
   estocasticidad.
3. **Fuga de datos** — la búsqueda de hiperparámetros ve SOLO train; val se usa únicamente
   para **elegir** entre candidatos ya ajustados (no se reentrena con val); test se toca
   **UNA sola vez** al final con el ganador.
4. **Shock / Δs** — máscara P75 ya validada: Sutil **23.2%**, Dulce **33.9%**, sobre
   `|Δy/y|·100` en **toneladas reales**, solo test. `Δs = (MAE_shock − MAE_global)/MAE_global × 100`.
5. **Dataset** — serie **cruda sin escalar** `master_dataset_{sutil,dulce}_v2.csv`
   (`produccion_t_{sutil|dulce}`), en toneladas. El escalado queda reservado para GE/GM.
6. **Búsqueda** — SARIMA grid `(p,d,q)(P,D,Q,12)` = `p 0..3, d 0..1, q 0..3, P 0..2, D 0..1,
   Q 0..2`; Prophet grid `changepoint_prior_scale ∈ {0.01,0.05,0.5}` × `fourier_order ∈ {4,6,10}`
   con estacionalidad anual `period=365.25 días` (Prophet mide el ciclo en días).

### Selección de modelo (documentación explícita)
`seleccion_metodo: "top-5 candidatos por AIC en train, seleccion final por menor MAE en
val (NO reentrenamiento con val, solo evaluacion de candidatos ya ajustados). Test tocado
una unica vez con el modelo ganador."`

## Entrada
- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2.csv` (120×18)
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2.csv` (120×18)

## Salida (trazabilidad)
- `v2_reentrenamiento/experimentos/exp_002_sarima_sutil|dulce/{config.yaml, metricas.json, predicciones.csv}`
- `v2_reentrenamiento/experimentos/exp_003_prophet_sutil|dulce/{config.yaml, metricas.json, predicciones.csv}`
- `v2_reentrenamiento/experimentos/REGISTRO_MAESTRO.csv` (+4 filas)
- `v2_reentrenamiento/resultados_v2_final/gc1_comparativa_naive_sarima_prophet.csv`


## 1. Configuración inicial


In [1]:
import os, json, logging, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
pd.set_option('display.width', 260)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

# Determinismo global (respuesta #2): seed fija ANTES de cualquier ajuste
SEED = 42
np.random.seed(SEED)

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
print('Raiz del proyecto:', os.getcwd())

# ---- Rutas ----
PROC     = 'v2_reentrenamiento/data/processed'
EXP_ROOT = 'v2_reentrenamiento/experimentos'
OUT_R2   = 'v2_reentrenamiento/resultados_v2_final'
os.makedirs(OUT_R2, exist_ok=True)

# ---- Umbrales P75 YA VALIDADOS (variación % en toneladas reales) ----
P75 = {'sutil': 23.2, 'dulce': 33.9}

# ---- Split ----
N_TRAIN, N_VAL, N_TEST = 90, 12, 12
FECHA = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')

COL_TARGET = {'sutil': 'produccion_t_sutil', 'dulce': 'produccion_t_dulce'}

SELECCION_METODO = ('top-5 candidatos por AIC en train, seleccion final por menor MAE en val '
                    '(NO reentrenamiento con val, solo evaluacion de candidatos ya ajustados). '
                    'Test tocado una unica vez con el modelo ganador.')
print('seed=', SEED, '| P75=', P75, '| split', N_TRAIN, N_VAL, N_TEST)


Raiz del proyecto: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
seed= 42 | P75= {'sutil': 23.2, 'dulce': 33.9} | split 90 12 12


## 2. Carga y particionado cronológico de la serie real (sin escalar)


In [2]:
def cargar(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df['fecha'] = df['año'].astype(str) + '-' + df['mes'].astype(str).str.zfill(2)
    df = df.sort_values(['año', 'mes']).reset_index(drop=True)
    df['participacion'] = 'buffer'
    df.loc[(df['fecha'] >= '2016-07') & (df['fecha'] <= '2023-12'), 'participacion'] = 'train'
    df.loc[df['fecha'] >= '2024-01', 'participacion'] = 'val'
    df.loc[df['fecha'] >= '2025-01', 'participacion'] = 'test'
    return df

def particiones(df):
    train = df[(df['fecha'] >= '2016-07') & (df['fecha'] <= '2023-12')].reset_index(drop=True)
    val   = df[(df['fecha'] >= '2024-01') & (df['fecha'] <= '2024-12')].reset_index(drop=True)
    test  = df[(df['fecha'] >= '2025-01') & (df['fecha'] <= '2025-12')].reset_index(drop=True)
    assert (len(train), len(val), len(test)) == (N_TRAIN, N_VAL, N_TEST)
    return train, val, test

series = {'sutil': cargar(f'{PROC}/master_dataset_sutil_v2.csv'),
          'dulce': cargar(f'{PROC}/master_dataset_dulce_v2.csv')}
for k in series:
    tr, va, te = particiones(series[k])
    print(f'{k.upper()}: train {len(tr)} ({tr.fecha.iloc[0]}..{tr.fecha.iloc[-1]}) | '
          f'val {len(va)} | test {len(te)} | buffer {int((series[k].participacion=="buffer").sum())}')


SUTIL: train 90 (2016-07..2023-12) | val 12 | test 12 | buffer 6
DULCE: train 90 (2016-07..2023-12) | val 12 | test 12 | buffer 6


In [3]:
def metricas(real, pred):
    real = np.asarray(real, dtype=float); pred = np.asarray(pred, dtype=float)
    mask = ~(np.isnan(real) | np.isnan(pred))
    real, pred = real[mask], pred[mask]
    return {'mae':  float(mean_absolute_error(real, pred)),
            'rmse': float(np.sqrt(mean_squared_error(real, pred))),
            'r2':   float(r2_score(real, pred))}


## 3. SARIMA — búsqueda de hiperparámetros (grid determinista sobre train)

Grid completo `(p,d,q)(P,D,Q,12)` con límites aprobados. Cada candidato se ajusta en train,
se pone su **AIC de train** y se guardan los **top-5**. La selección final (menor MAE val)
se hace en la sección siguiente con los modelos YA ajustados, sin reentrenar.


In [4]:
PS_RANGE = range(0, 4)   # p: 0..3
DS_RANGE = range(0, 2)   # d: 0..1
QS_RANGE = range(0, 4)   # q: 0..3
PS_SEAS  = range(0, 3)   # P: 0..2
DS_SEAS  = range(0, 2)   # D: 0..1
QS_SEAS  = range(0, 3)   # Q: 0..2
M_SEASONAL = 12

def grid_sarima_train(y, maxiter=200):
    """Ajusta todos los candidatos sobre train; devuelve DataFrame base + modelos ajustados."""
    filas, modelos = [], {}
    for p in PS_RANGE:
        for d in DS_RANGE:
            for q in QS_RANGE:
                for P in PS_SEAS:
                    for D in DS_SEAS:
                        for Q in QS_SEAS:
                            orden = (p, d, q)
                            s_ord = (P, D, Q, M_SEASONAL)
                            try:
                                fit = SARIMAX(y, order=orden, seasonal_order=s_ord,
                                              enforce_stationarity=False,
                                              enforce_invertibility=False,
                                              initialization='approximate_diffuse',
                                              trend='c').fit(disp=False, maxiter=maxiter)
                                modelos[(p, d, q, P, D, Q)] = fit
                                filas.append((p, d, q, P, D, Q, float(fit.aic)))
                            except Exception:
                                continue
    df = pd.DataFrame(filas, columns=['p', 'd', 'q', 'P', 'D', 'Q', 'aic'])
    return df.sort_values('aic').reset_index(drop=True), modelos

candidatos = {}
for k in COL_TARGET:
    y = series[k].loc[series[k].participacion == 'train', COL_TARGET[k]].astype(float).to_numpy()
    df_g, mods = grid_sarima_train(y)
    candidatos[k] = {'grid': df_g, 'modelos': mods}
    print(f'=== {k.upper()} | candidatos ajustados (train, AIC <= {len(df_g)}) ===')
    print(df_g.head(5).to_string(index=False))


=== SUTIL | candidatos ajustados (train, AIC <= 576) ===
 p  d  q  P  D  Q        aic
 1  1  2  2  1  1 1,872.8306
 1  1  2  2  1  0 1,873.9669
 2  1  2  2  1  1 1,874.8893
 1  1  3  2  1  0 1,874.9307
 1  1  2  2  1  2 1,874.9919


=== DULCE | candidatos ajustados (train, AIC <= 576) ===
 p  d  q  P  D  Q      aic
 1  0  0  0  1  0 911.1554
 2  0  0  0  1  0 911.4875
 1  0  1  0  1  0 911.7000
 2  0  1  0  1  0 913.4865
 0  0  3  1  0  0 914.0954


## 4. SARIMA — selección final por menor MAE en val (candidatos ya ajustados)

De los **top-5 por AIC** se pronostican 12 pasos (perspectiva de val) con el modelo YA
ajustado en train (sin reentrenamiento) y se elige el de **menor MAE_val**. Esa elección
queda fijada; test nunca se observa aquí.


In [5]:
top_n = 5
seleccion_sarima = {}
for k in COL_TARGET:
    df_g  = candidatos[k]['grid'].head(top_n)
    mods  = candidatos[k]['modelos']
    val_y = series[k].loc[series[k].participacion == 'val', COL_TARGET[k]].astype(float).to_numpy()
    rows  = []
    for _, r in df_g.iterrows():
        fit = mods[(int(r.p), int(r.d), int(r.q), int(r.P), int(r.D), int(r.Q))]
        pred = np.asarray(fit.forecast(steps=N_VAL))
        rows.append({'order_str': f'({int(r.p)},{int(r.d)},{int(r.q)})({int(r.P)},{int(r.D)},{int(r.Q)},12)',
                     'order':     (int(r.p), int(r.d), int(r.q), int(r.P), int(r.D), int(r.Q)),
                     'aic_train': r.aic,
                     'mae_val':   float(mean_absolute_error(val_y, pred))})
    sel = pd.DataFrame(rows).sort_values('mae_val').reset_index(drop=True)
    seleccion_sarima[k] = sel.iloc[0].to_dict()
    print(f'=== {k.upper()} — TOP-5 por AIC, seleccion por MAE_val ===')
    print(sel[['order_str', 'aic_train', 'mae_val']].to_string(index=False))
    print(f'  >> SELECCIONADO: {sel.iloc[0]["order_str"]} (aic_train={sel.iloc[0]["aic_train"]:.2f}, '
          f'mae_val={sel.iloc[0]["mae_val"]:.2f})')


=== SUTIL — TOP-5 por AIC, seleccion por MAE_val ===
        order_str  aic_train    mae_val
(1,1,3)(2,1,0,12) 1,874.9307 8,439.7942
(1,1,2)(2,1,0,12) 1,873.9669 8,609.5266
(1,1,2)(2,1,2,12) 1,874.9919 9,181.7774
(1,1,2)(2,1,1,12) 1,872.8306 9,301.9980
(2,1,2)(2,1,1,12) 1,874.8893 9,408.5844
  >> SELECCIONADO: (1,1,3)(2,1,0,12) (aic_train=1874.93, mae_val=8439.79)
=== DULCE — TOP-5 por AIC, seleccion por MAE_val ===
        order_str  aic_train  mae_val
(1,0,0)(0,1,0,12)   911.1554  38.0560
(2,0,0)(0,1,0,12)   911.4875  38.6252
(2,0,1)(0,1,0,12)   913.4865  38.6678
(1,0,1)(0,1,0,12)   911.7000  39.2269
(0,0,3)(1,0,0,12)   914.0954  39.8337
  >> SELECCIONADO: (1,0,0)(0,1,0,12) (aic_train=911.16, mae_val=38.06)


## 5. SARIMA — evaluación final (train in-sample, val y test out-of-sample)

El orden ganador se **reajusta sobre train** (mismos datos, MLE determinista) y se producen:
- **train:** predicción in-sample (`fittedvalues` por `predict(0..89)`).
- **val:** forecast de 12 pasos (out-of-sample).
- **test:** forecast de 12 pasos — **único toque de test**, con el ganador ya fijado.
Se reportan MAE/RMSE/R² por partición y `Δs` sobre test con la máscara P75.


In [6]:
sarima_full = {}
for k in COL_TARGET:
    serie  = series[k]
    tr, va, te = particiones(serie)
    y = tr[COL_TARGET[k]].astype(float).to_numpy()

    order6  = seleccion_sarima[k]['order']
    order_s = seleccion_sarima[k]['order_str']
    ar      = tuple(order6[:3])
    sr      = tuple(order6[3:]) + (12,)
    fit = SARIMAX(y, order=ar, seasonal_order=sr,
                  enforce_stationarity=False, enforce_invertibility=False,
                  initialization='approximate_diffuse', trend='c').fit(disp=False, maxiter=500)

    # HORIZONTE REAL (corregido): val son los pasos 1..12 desde el fin de train y
    # test los pasos 13..24. Antes ambas lineas llamaban a forecast() desde el mismo
    # punto, por lo que la prediccion de test replicaba exactamente la de val.
    f_val_test = np.asarray(fit.forecast(steps=N_VAL + N_TEST))
    pred_train = np.asarray(fit.predict(start=0, end=N_TRAIN - 1))
    pred_val   = f_val_test[:N_VAL]
    pred_test  = f_val_test[N_VAL:]

    nombres = {'train': (tr[COL_TARGET[k]].to_numpy(), pred_train),
               'val':   (va[COL_TARGET[k]].to_numpy(), pred_val),
               'test':  (te[COL_TARGET[k]].to_numpy(), pred_test)}
    metrics = {p: metricas(*nombres[p]) for p in nombres}

    # Series de prediccion alineadas con el dataset completo (para predicciones.csv)
    pred_series = np.full(len(serie), np.nan)
    pred_series[(serie.participacion == 'train').to_numpy()] = pred_train
    pred_series[(serie.participacion == 'val').to_numpy()]   = pred_val
    pred_series[(serie.participacion == 'test').to_numpy()]  = pred_test
    serie_aux = serie.copy(); serie_aux['predicho_t'] = np.round(pred_series, 2)

    # Δs sobre test (máscara P75 ya validada)
    serie_aux['shock'] = False
    serie_aux.loc[serie_aux.participacion == 'test', 'shock'] =         (100 * serie_aux[COL_TARGET[k]].pct_change().abs() > P75[k])[serie_aux.participacion == 'test'].to_numpy()
    test = serie_aux[serie_aux.participacion == 'test'].copy()
    shocks = test[test['shock']]
    mae_global = metrics['test']['mae']
    if len(shocks) > 0:
        mae_shock = float(mean_absolute_error(shocks[COL_TARGET[k]], shocks['predicho_t']))
        ds = 100.0 * (mae_shock - mae_global) / mae_global
    else:
        mae_shock, ds = float('nan'), float('nan')
    metrics['test'].update({'n_shock': int(len(shocks)), 'mae_shock': mae_shock, 'delta_s_pct': ds})

    sarima_full[k] = {'serie': serie_aux, 'metrics': metrics, 'order': order_s,
                      'aic_train': seleccion_sarima[k]['aic_train'], 'mae_val': seleccion_sarima[k]['mae_val']}
    print('=' * 70)
    print(f'SARIMA — {k.upper()} | orden {order_s} | aic_train {seleccion_sarima[k]["aic_train"]:.2f}')
    print('=' * 70)
    for p in ('train', 'val', 'test'):
        m = metrics[p]
        print(f'  {p.upper():5s} | MAE = {m["mae"]:12,.2f} | RMSE = {m["rmse"]:12,.2f} | R² = {m["r2"]:+.4f}')
    if metrics['test']['n_shock']:
        print(f'  TEST shock: n={metrics["test"]["n_shock"]} | MAE_shock = {metrics["test"]["mae_shock"]:,.2f} | '
              f'Δs = {metrics["test"]["delta_s_pct"]:+.2f}%')
    else:
        print(f'  TEST: sin shocks segun P75 {P75[k]}% (Δs no definido)')


SARIMA — SUTIL | orden (1,1,3)(2,1,0,12) | aic_train 1874.93
  TRAIN | MAE =     3,110.40 | RMSE =     4,350.58 | R² = +0.7073
  VAL   | MAE =     8,439.79 | RMSE =     9,398.86 | R² = -0.3183
  TEST  | MAE =     8,139.30 | RMSE =     9,128.16 | R² = -0.0969
  TEST shock: n=3 | MAE_shock = 8,023.51 | Δs = -1.42%
SARIMA — DULCE | orden (1,0,0)(0,1,0,12) | aic_train 911.16
  TRAIN | MAE =        62.04 | RMSE =       142.61 | R² = +0.0220
  VAL   | MAE =        38.06 | RMSE =        47.06 | R² = +0.9072
  TEST  | MAE =        57.57 | RMSE =        61.53 | R² = +0.8686
  TEST shock: n=3 | MAE_shock = 55.82 | Δs = -3.03%


## 6. Prophet — búsqueda de hiperparámetros (grid sobre train)

Grid aprobado: `changepoint_prior_scale ∈ {0.01, 0.05, 0.5}` × `fourier_order ∈ {4, 6, 10}`.
Estacionalidad anual vía `period=365.25` días (Prophet mide ciclos en días; "año" = 365.25,
no 12). `weekly=False`, `daily=False`, `mcmc_samples=0`, `growth='linear'`,
`seasonality_mode='additive'`. Cada candidato se ajusta en train y se elige por menor MAE_val
(candidatos ya ajustados, sin reentrenamiento con val).


In [7]:
CPS_GRID  = [0.01, 0.05, 0.5]
FO_GRID   = [4, 6, 10]

def fit_prophet(y_train, fecha_train, fo, cps):
    modelo = Prophet(growth='linear', seasonality_mode='additive',
                     changepoint_prior_scale=cps,
                     weekly_seasonality=False, daily_seasonality=False,
                     yearly_seasonality=False, mcmc_samples=0)
    modelo.add_seasonality(name='yearly_12', period=365.25, fourier_order=int(fo))
    d = pd.DataFrame({'ds': pd.to_datetime(fecha_train), 'y': y_train})
    modelo.fit(d)
    return modelo

def prophet_forecast(modelo, fecha_horizonte, n):
    fut = pd.DataFrame({'ds': pd.date_range(pd.to_datetime(fecha_horizonte), periods=n, freq='MS')})
    return np.asarray(modelo.predict(fut)['yhat'])

prophet_grid = {}
for k in COL_TARGET:
    tr = series[k].loc[series[k].participacion == 'train']
    val_y = series[k].loc[series[k].participacion == 'val', COL_TARGET[k]].astype(float).to_numpy()
    filas = []
    for cps in CPS_GRID:
        for fo in FO_GRID:
            m = fit_prophet(tr[COL_TARGET[k]].astype(float).to_numpy(),
                            tr.fecha.values, fo, cps)
            pred = prophet_forecast(m, '2024-01-01', N_VAL)
            filas.append({'changepoint_prior_scale': cps, 'fourier_order': fo,
                          'mae_val': float(mean_absolute_error(val_y, pred)),
                          'modelo': m})
    sel_df = pd.DataFrame([{k2: v for k2, v in f.items() if k2 != 'modelo'} for f in filas])
    sel_df = sel_df.sort_values('mae_val').reset_index(drop=True)
    best = sel_df.iloc[0]
    modelos = {f['changepoint_prior_scale']: {} for f in filas}
    for f in filas:
        modelos[f['changepoint_prior_scale']][f['fourier_order']] = f['modelo']
    prophet_grid[k] = {'tabla': sel_df, 'mejor': best.to_dict(), 'modelos': modelos}
    print(f'=== {k.upper()} — Prophet grid (9 candidatos), seleccion por MAE_val ===')
    print(sel_df.to_string(index=False))
    print(f'  >> SELECCIONADO: cps={best["changepoint_prior_scale"]}, '
          f'fourier_order={best["fourier_order"]} (mae_val={best["mae_val"]:.2f})')


22:48:19 - cmdstanpy - INFO - Chain [1] start processing


22:48:19 - cmdstanpy - INFO - Chain [1] done processing


22:48:19 - cmdstanpy - INFO - Chain [1] start processing


22:48:19 - cmdstanpy - INFO - Chain [1] done processing


22:48:19 - cmdstanpy - INFO - Chain [1] start processing


...

22:48:24 - cmdstanpy - INFO - Chain [1] start processing


22:48:24 - cmdstanpy - INFO - Chain [1] done processing


22:48:24 - cmdstanpy - INFO - Chain [1] start processing


22:48:25 - cmdstanpy - INFO - Chain [1] done processing


=== DULCE — Prophet grid (9 candidatos), seleccion por MAE_val ===
 changepoint_prior_scale  fourier_order  mae_val
                  0.0500              6  46.8896
                  0.0500              4  47.7959
                  0.0100              6  47.8332
                  0.0100              4  47.9880
                  0.0500             10  52.7302
                  0.5000              4  53.4193
                  0.0100             10  53.4602
                  0.5000              6  53.4672
                  0.5000             10  59.8861
  >> SELECCIONADO: cps=0.05, fourier_order=6.0 (mae_val=46.89)


In [8]:
prophet_full = {}
for k in COL_TARGET:
    serie = series[k]
    tr, va, te = particiones(serie)
    cps = float(prophet_grid[k]['mejor']['changepoint_prior_scale'])
    fo  = int(prophet_grid[k]['mejor']['fourier_order'])
    m = fit_prophet(tr[COL_TARGET[k]].astype(float).to_numpy(), tr.fecha.values, fo, cps)

    # train in-sample + val/test out-of-sample
    pred_train = np.asarray(m.predict(pd.DataFrame({'ds': pd.to_datetime(tr.fecha.values)}))['yhat'])
    pred_val   = prophet_forecast(m, '2024-01-01', N_VAL)
    pred_test  = prophet_forecast(m, '2025-01-01', N_TEST)

    nombres = {'train': (tr[COL_TARGET[k]].to_numpy(), pred_train),
               'val':   (va[COL_TARGET[k]].to_numpy(), pred_val),
               'test':  (te[COL_TARGET[k]].to_numpy(), pred_test)}
    metrics = {p: metricas(*nombres[p]) for p in nombres}

    pred_series = np.full(len(serie), np.nan)
    pred_series[(serie.participacion == 'train').to_numpy()] = pred_train
    pred_series[(serie.participacion == 'val').to_numpy()]   = pred_val
    pred_series[(serie.participacion == 'test').to_numpy()]  = pred_test
    serie_aux = serie.copy(); serie_aux['predicho_t'] = np.round(pred_series, 2)

    serie_aux['shock'] = False
    serie_aux.loc[serie_aux.participacion == 'test', 'shock'] =         (100 * serie_aux[COL_TARGET[k]].pct_change().abs() > P75[k])[serie_aux.participacion == 'test'].to_numpy()
    test = serie_aux[serie_aux.participacion == 'test'].copy()
    shocks = test[test['shock']]
    mae_global = metrics['test']['mae']
    if len(shocks) > 0:
        mae_shock = float(mean_absolute_error(shocks[COL_TARGET[k]], shocks['predicho_t']))
        ds = 100.0 * (mae_shock - mae_global) / mae_global
    else:
        mae_shock, ds = float('nan'), float('nan')
    metrics['test'].update({'n_shock': int(len(shocks)), 'mae_shock': mae_shock, 'delta_s_pct': ds})

    prophet_full[k] = {'serie': serie_aux, 'metrics': metrics,
                       'cps': float(cps), 'fo': int(fo), 'mae_val': prophet_grid[k]['mejor']['mae_val']}
    print('=' * 70)
    print(f'PROPHET — {k.upper()} | cps={cps}, fourier_order={fo}')
    print('=' * 70)
    for p in ('train', 'val', 'test'):
        m = metrics[p]
        print(f'  {p.upper():5s} | MAE = {m["mae"]:12,.2f} | RMSE = {m["rmse"]:12,.2f} | R² = {m["r2"]:+.4f}')
    if metrics['test']['n_shock']:
        print(f'  TEST shock: n={metrics["test"]["n_shock"]} | MAE_shock = {metrics["test"]["mae_shock"]:,.2f} | '
              f'Δs = {metrics["test"]["delta_s_pct"]:+.2f}%')
    else:
        print(f'  TEST: sin shocks segun P75 {P75[k]}% (Δs no definido)')


22:48:25 - cmdstanpy - INFO - Chain [1] start processing


22:48:25 - cmdstanpy - INFO - Chain [1] done processing


22:48:25 - cmdstanpy - INFO - Chain [1] start processing


PROPHET — SUTIL | cps=0.01, fourier_order=4
  TRAIN | MAE =     3,887.30 | RMSE =     4,817.18 | R² = +0.6412
  VAL   | MAE =     3,796.39 | RMSE =     4,628.33 | R² = +0.6803
  TEST  | MAE =     3,682.03 | RMSE =     3,970.57 | R² = +0.7925
  TEST shock: n=3 | MAE_shock = 3,016.42 | Δs = -18.08%


22:48:25 - cmdstanpy - INFO - Chain [1] done processing


PROPHET — DULCE | cps=0.05, fourier_order=6
  TRAIN | MAE =        22.90 | RMSE =        29.00 | R² = +0.9596
  VAL   | MAE =        46.89 | RMSE =        55.31 | R² = +0.8718
  TEST  | MAE =        64.58 | RMSE =        74.09 | R² = +0.8094
  TEST shock: n=3 | MAE_shock = 91.51 | Δs = +41.70%


## 7. Registro de experimentos en el sistema de trazabilidad v2

Se crean 4 carpetas (`exp_002_sarima_sutil|dulce`, `exp_003_prophet_sutil|dulce`) con
`config.yaml` (incluye el campo **`seleccion_metodo`** aprobado), `metricas.json` y
`predicciones.csv`.


In [9]:
import yaml

DATA_SPLIT = {
    'train': {'rango': '2016-07..2023-12', 'meses': N_TRAIN},
    'val':   {'rango': '2024-01..2024-12', 'meses': N_VAL},
    'test':  {'rango': '2025-01..2025-12', 'meses': N_TEST},
}
SHOCK_CFG = {
    'definicion': '|100*(y_t - y_{t-1})/y_{t-1}| > P75',
    'escala': 'serie real en toneladas (NUNCA z-score)',
    'umbral_p75_pct': None,
    'sobre': 'TEST unicamente',
}
DELTA_CFG = {
    'formula': '(MAE_shock - MAE_global) / MAE_global * 100',
    'observacion': 'deterioro relativo del modelo en meses de shock vs todo el test',
}
ENV = {'python': '3.11', 'seed': SEED, 'fecha_ejecucion': FECHA}

def escribir_exp(exp_id, cultivo, modelo, cfg_extra, metrics, serie_aux, pred_col, shock_col):
    out_dir = os.path.join(EXP_ROOT, exp_id)
    os.makedirs(out_dir, exist_ok=True)
    cfg = {
        'experimento': exp_id,
        'cultivo': cultivo.upper(),
        'modelo': modelo,
        'descripcion': 'GC1 univariado (solo serie de produccion propia, sin regresores externos)',
        'seleccion_metodo': SELECCION_METODO,
        'estado': 'ejecutado',
        'datos': {
            'entrada': f'master_dataset_{cultivo}_v2.csv',
            'fuente': 'Midagri (produccion_t)',
            'unidades': 'toneladas (serie cruda, sin StandardScaler)',
            'n_filas': int(len(serie_aux)),
            'rango': f'{serie_aux.fecha.iloc[0]}..{serie_aux.fecha.iloc[-1]}',
            'nulos': 0,
        },
        'split': DATA_SPLIT,
        'shock': {**SHOCK_CFG, 'umbral_p75_pct': P75[cultivo]},
        'delta_s': DELTA_CFG,
        'modelo_ganador': cfg_extra,
        'entorno': ENV,
    }
    with open(os.path.join(out_dir, 'config.yaml'), 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False, default_flow_style=False)
    with open(os.path.join(out_dir, 'metricas.json'), 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    seria = serie_aux.copy()
    pred_col_real = 'predicho_t'
    out = pd.DataFrame({
        'fecha':      seria.fecha,
        'real_t':     seria[COL_TARGET[cultivo]].round(2),
        'predicho_t': seria[pred_col_real],
        'particion':  seria.participacion,
        'shock_test': seria['shock'].astype(bool),
    })
    out.to_csv(os.path.join(out_dir, 'predicciones.csv'), index=False, encoding='utf-8-sig')
    return out_dir

agentes = [
    ('exp_002_sarima_sutil',  'sutil', 'SARIMA',  {'order': sarima_full['sutil']['order'],
                                                    'aic_train': round(sarima_full['sutil']['aic_train'], 2),
                                                    'mae_val': round(sarima_full['sutil']['mae_val'], 2)}),
    ('exp_002_sarima_dulce',  'dulce', 'SARIMA',  {'order': sarima_full['dulce']['order'],
                                                    'aic_train': round(sarima_full['dulce']['aic_train'], 2),
                                                    'mae_val': round(sarima_full['dulce']['mae_val'], 2)}),
    ('exp_003_prophet_sutil', 'sutil', 'Prophet', {'changepoint_prior_scale': float(prophet_full['sutil']['cps']),
                                                    'fourier_order': int(prophet_full['sutil']['fo']),
                                                    'seasonality_anual_period_dias': 365.25,
                                                    'mae_val': round(prophet_full['sutil']['mae_val'], 2)}),
    ('exp_003_prophet_dulce', 'dulce', 'Prophet', {'changepoint_prior_scale': float(prophet_full['dulce']['cps']),
                                                    'fourier_order': int(prophet_full['dulce']['fo']),
                                                    'seasonality_anual_period_dias': 365.25,
                                                    'mae_val': round(prophet_full['dulce']['mae_val'], 2)}),
]

for exp_id, cultivo, modelo, cfg_extra in agentes:
    info = sarima_full[cultivo] if modelo == 'SARIMA' else prophet_full[cultivo]
    out_dir = escribir_exp(exp_id, cultivo, modelo, cfg_extra, info['metrics'],
                           info['serie'], 'predicho_t', 'shock')
    print(f'[OK] {exp_id} techo guardado en {out_dir}')


[OK] exp_002_sarima_sutil techo guardado en v2_reentrenamiento/experimentos\exp_002_sarima_sutil
[OK] exp_002_sarima_dulce techo guardado en v2_reentrenamiento/experimentos\exp_002_sarima_dulce
[OK] exp_003_prophet_sutil techo guardado en v2_reentrenamiento/experimentos\exp_003_prophet_sutil
[OK] exp_003_prophet_dulce techo guardado en v2_reentrenamiento/experimentos\exp_003_prophet_dulce


## 8. Registro maestro de experimentos (`REGISTRO_MAESTRO.csv` +4 filas)


In [10]:
REG = os.path.join(EXP_ROOT, 'REGISTRO_MAESTRO.csv')
COLUMNAS = ['experimento', 'cultivo', 'modelo', 'estado', 'datos', 'split',
            'umbral_p75', 'mae_train', 'mae_val', 'mae_test',
            'rmse_train', 'rmse_val', 'rmse_test',
            'r2_train', 'r2_val', 'r2_test',
            'n_shocks_test', 'mae_shock_test', 'delta_s_pct', 'fecha']

filas = []
for exp_id, cultivo, modelo, cfg_extra in agentes:
    info = sarima_full[cultivo] if modelo == 'SARIMA' else prophet_full[cultivo]
    m    = info['metrics']
    t    = m['test']
    filas.append({
        'experimento':    exp_id,
        'cultivo':        cultivo.upper(),
        'modelo':         modelo,
        'estado':         'ejecutado',
        'datos':          f'master_dataset_{cultivo}_v2.csv (120x18, crudo)',
        'split':          'train 2016-07..2023-12 (90) | val 2024 (12) | test 2025 (12)',
        'umbral_p75':     P75[cultivo],
        'mae_train':      round(m['train']['mae'], 2),
        'mae_val':        round(m['val']['mae'],   2),
        'mae_test':       round(m['test']['mae'],  2),
        'rmse_train':     round(m['train']['rmse'], 2),
        'rmse_val':       round(m['val']['rmse'],   2),
        'rmse_test':      round(m['test']['rmse'],  2),
        'r2_train':       round(m['train']['r2'], 4),
        'r2_val':         round(m['val']['r2'],     4),
        'r2_test':        round(m['test']['r2'],    4),
        'n_shocks_test':  t['n_shock'],
        'mae_shock_test': None if t['n_shock'] == 0 else round(t['mae_shock'], 2),
        'delta_s_pct':    None if t['n_shock'] == 0 else round(t['delta_s_pct'], 2),
        'fecha':          FECHA,
    })

df_reg = pd.DataFrame(filas, columns=COLUMNAS)
if os.path.exists(REG):
    prev = pd.read_csv(REG, encoding='utf-8-sig')
    df_reg = pd.concat([prev, df_reg], ignore_index=True)
df_reg = df_reg.drop_duplicates(subset=['experimento'], keep='last')
os.makedirs(EXP_ROOT, exist_ok=True)
df_reg.to_csv(REG, index=False, encoding='utf-8-sig')
print('REGISTRO_MAESTRO.csv actualizado ->', len(df_reg), 'filas')
print(df_reg[['experimento', 'cultivo', 'modelo', 'mae_test', 'delta_s_pct']].to_string(index=False))


REGISTRO_MAESTRO.csv actualizado -> 6 filas
          experimento cultivo  modelo   mae_test  delta_s_pct
  exp_001_naive_sutil   SUTIL   Naive 4,704.2900     147.8600
  exp_001_naive_dulce   DULCE   Naive    84.7000      63.2300
 exp_002_sarima_sutil   SUTIL  SARIMA 8,139.3000      -1.4200
 exp_002_sarima_dulce   DULCE  SARIMA    57.5700      -3.0300
exp_003_prophet_sutil   SUTIL Prophet 3,682.0300     -18.0800
exp_003_prophet_dulce   DULCE Prophet    64.5800      41.7000


## 9. Comparativa GC1 — Naive vs SARIMA vs Prophet

Se cargan las métricas del Naive (`exp_001_*`) y se arma la tabla final con
MAE / RMSE / R² (train, val, test) y Δs (test) para ambos productos.


In [ ]:
# Mapeo explicito modelo -> experimento para la comparativa oficial GC1.
# SARIMA-Sutil usa exp_002b (seleccion MANUAL por parsimonia), NO exp_002
# (seleccion automatica por AIC: mejor AIC en train pero R2_test = -1.11).
# Ver DECISIONES_METODOLOGICAS.md - "Sobreajuste por AIC en SARIMA-Sutil".
EXP_COMPARATIVA = {
    ('Naive',   'sutil'): 'exp_001_naive_sutil',
    ('SARIMA',  'sutil'): 'exp_002b_sarima_sutil_simple',
    ('Prophet', 'sutil'): 'exp_003_prophet_sutil',
    ('Naive',   'dulce'): 'exp_001_naive_dulce',
    ('SARIMA',  'dulce'): 'exp_002_sarima_dulce',
    ('Prophet', 'dulce'): 'exp_003_prophet_dulce',
}


def lee_metrics(exp_id):
    ruta = os.path.join(EXP_ROOT, exp_id, 'metricas.json')
    if not os.path.exists(ruta):
        raise FileNotFoundError(
            f'Falta {ruta}. La comparativa GC1 depende de '
            'exp_002b_sarima_sutil_simple, que genera '
            '02b_diagnostico_sarima_sutil.ipynb. '
            'Ejecuta ese notebook antes que esta celda.')
    with open(ruta, encoding='utf-8') as f:
        return json.load(f)


comparativa = []
for cultivo in ('sutil', 'dulce'):
    for modelo in ('Naive', 'SARIMA', 'Prophet'):
        exp_id = EXP_COMPARATIVA[(modelo, cultivo)]
        m = lee_metrics(exp_id)
        for part in ('train', 'val', 'test'):
            comparativa.append({
                'cultivo': cultivo.upper(), 'modelo': modelo, 'particion': part,
                'mae':  m[part]['mae'], 'rmse': m[part]['rmse'], 'r2': m[part]['r2'],
                'delta_s': m['test'].get('delta_s_pct') if part == 'test' else None,
                'experimento': exp_id,
            })
cmp_df = pd.DataFrame(comparativa)
cmp_out = os.path.join(OUT_R2, 'gc1_comparativa_naive_sarima_prophet.csv')
cmp_df.to_csv(cmp_out, index=False, encoding='utf-8-sig')
print('Guardado:', cmp_out)
print()

for cultivo in ('SUTIL', 'DULCE'):
    sub = cmp_df[cmp_df.cultivo == cultivo]
    piv = sub.pivot(index='modelo', columns='particion', values=['mae', 'rmse', 'r2'])
    print('=' * 84)
    print(cultivo)
    print('=' * 84)
    for var, unidad in [('mae', 'MAE'), ('rmse', 'RMSE')]:
        print(f'  {unidad}:')
        print(piv[var].round(2).to_string())
    print('  R2:')
    print(piv['r2'].round(4).to_string())
    print()
    t = sub[sub.particion == 'test'][['modelo', 'mae', 'delta_s']].set_index('modelo').round(2)
    print('  TEST (MAE y Δs):')
    print(t.to_string())

# sobrepasa al Naive en MAE_test global
print()
print('=' * 84)
print('¿GC1 supera al Naive en MAE_test global?   (menor MAE = mejor)')
print('=' * 84)
for cultivo in ('SUTIL', 'DULCE'):
    t = cmp_df[(cmp_df.cultivo == cultivo) & (cmp_df.particion == 'test')][['modelo', 'mae']]
    naive = t[t.modelo == 'Naive'].iloc[0].mae
    for _, r in t.iterrows():
        delta = (r.mae - naive) / naive * 100
        flag = 'SUPERA al Naive' if r.mae < naive else 'NO supera al Naive'
        print(f'  {cultivo:5s} {r.modelo:7s} MAE_test={r.mae:10,.2f}  vs Naive {naive:10,.2f}  '
              f'({delta:+6.1f}%)  {flag}')


## 10. Conclusión GC1


In [12]:
print('CONCLUSION GC1 (a validar contra cota Naive):')
for cultivo in ('SUTIL', 'DULCE'):
    t = cmp_df[(cmp_df.cultivo == cultivo) & (cmp_df.particion == 'test')][['modelo', 'mae', 'delta_s']].set_index('modelo').round(2)
    print()
    print(cultivo)
    print(t.to_string())
print()
print('=== FIN ACTIVIDAD 02 — GC1 (SARIMA + Prophet) v2 ===')
print('Siguiente paso: SARIMAX+LSTM (GC2) — híbrido clásico+DL.')


CONCLUSION GC1 (a validar contra cota Naive):

SUTIL
               mae  delta_s
modelo                     
Naive   4,704.2900 147.8600
SARIMA  8,139.3000  -1.4200
Prophet 3,682.0300 -18.0800

DULCE
            mae  delta_s
modelo                  
Naive   84.7000  63.2300
SARIMA  57.5700  -3.0300
Prophet 64.5800  41.7000

=== FIN ACTIVIDAD 02 — GC1 (SARIMA + Prophet) v2 ===
Siguiente paso: SARIMAX+LSTM (GC2) — híbrido clásico+DL.
